In [ ]:
!pip install -q transformers accelerate datasets evaluate sentencepiece protobuf
!pip install -q scikit-learn pandas numpy
!pip install -q torch torchvision torchaudio

import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import RobertaTokenizer, T5EncoderModel
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

df = pd.read_parquet(
    "hf://datasets/DaniilOr/CoDET-M4/dataset_without_comments.parquet"
)

df = df[
    [
        "cleaned_code",
        "language",
        "target",
        "split"
    ]
].copy()

df = df.dropna(subset=["cleaned_code", "language", "target", "split"])

df = df.drop_duplicates(subset=["cleaned_code"])

df = df[df["cleaned_code"].str.len() > 20]

df["label"] = df["target"].map({
    "human": 0,
    "ai": 1
})

print("Cleaned dataset shape:", df.shape)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"].isin(["validation", "valid", "val"])].copy()
test_df = df[df["split"] == "test"].copy()

print("Official train shape:", train_df.shape)
print("Official validation shape:", val_df.shape)
print("Official test shape:", test_df.shape)

reduced_train_parts = []

languages = ["python", "java", "cpp"]

for lang in languages:
    lang_train = train_df[train_df["language"] == lang]

    human_part = lang_train[lang_train["label"] == 0]
    ai_part = lang_train[lang_train["label"] == 1]

    n_samples = min(len(human_part), len(ai_part), 3000)

    sampled_human = human_part.sample(n=n_samples, random_state=42)
    sampled_ai = ai_part.sample(n=n_samples, random_state=42)

    reduced_train_parts.append(sampled_human)
    reduced_train_parts.append(sampled_ai)

train_df = pd.concat(reduced_train_parts)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Reduced train shape:", train_df.shape)
print(train_df["label"].value_counts())
print(train_df["language"].value_counts())

train_df = train_df[["cleaned_code", "label", "language"]]
val_df = val_df[["cleaned_code", "label", "language"]]
test_df = test_df[["cleaned_code", "label", "language"]]

AST_DIM = 7

def extract_ast_features(code, language):
    if language != "python":
        return [0.0] * AST_DIM

    try:
        tree = ast.parse(code)
        nodes = list(ast.walk(tree))

        return [
            float(sum(isinstance(n, ast.FunctionDef) for n in nodes)),
            float(sum(isinstance(n, ast.AsyncFunctionDef) for n in nodes)),
            float(sum(isinstance(n, ast.ClassDef) for n in nodes)),
            float(sum(isinstance(n, (ast.For, ast.While)) for n in nodes)),
            float(sum(isinstance(n, ast.If) for n in nodes)),
            float(sum(isinstance(n, ast.Assign) for n in nodes)),
            float(sum(isinstance(n, (ast.Import, ast.ImportFrom)) for n in nodes)),
        ]
    except:
        return [0.0] * AST_DIM

for dataset_df in [train_df, val_df, test_df]:
    dataset_df["ast_features"] = dataset_df.apply(
        lambda row: extract_ast_features(
            row["cleaned_code"],
            row["language"]
        ),
        axis=1
    )

    dataset_df["model_input"] = (
        "language: " + dataset_df["language"] +
        " code: " + dataset_df["cleaned_code"]
    )

print("AST features extracted.")

MODEL_NAME = "Salesforce/codet5-base"

tokenizer = RobertaTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    extra_special_tokens=[]
)

MAX_LEN = 256

class CodeDataset(TorchDataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoded = tokenizer(
            row["model_input"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "ast_features": torch.tensor(row["ast_features"], dtype=torch.float),
            "labels": torch.tensor(row["label"], dtype=torch.long)
        }

BATCH_SIZE = 4

train_loader = DataLoader(
    CodeDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    CodeDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    CodeDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

class CodeT5FusionModel(nn.Module):
    def __init__(self, encoder_hidden=768, ast_dim=AST_DIM):
        super().__init__()

        self.encoder = T5EncoderModel.from_pretrained(MODEL_NAME)

        self.ast_proj = nn.Sequential(
            nn.Linear(ast_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.fusion = nn.Sequential(
            nn.Linear(encoder_hidden + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.classifier = nn.Linear(128, 2)

    def forward(self, input_ids, attention_mask, ast_features):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        code_embedding = outputs.last_hidden_state[:, 0, :]
        ast_embedding = self.ast_proj(ast_features)

        fused = self.fusion(
            torch.cat([code_embedding, ast_embedding], dim=1)
        )

        logits = self.classifier(fused)

        return logits, fused

model_fusion = CodeT5FusionModel().to(DEVICE)

def contrastive_loss(embeddings, labels, margin=1.0):
    batch_size = embeddings.size(0)

    if batch_size < 2:
        return torch.tensor(0.0, device=DEVICE)

    diff = embeddings.unsqueeze(1) - embeddings.unsqueeze(0)
    dist = diff.norm(dim=-1)

    same_mask = (labels.unsqueeze(1) == labels.unsqueeze(0)).float()
    diff_mask = 1.0 - same_mask

    loss_same = same_mask * dist.pow(2)
    loss_diff = diff_mask * F.relu(margin - dist).pow(2)

    mask = 1.0 - torch.eye(batch_size, device=DEVICE)

    loss = ((loss_same + loss_diff) * mask).sum() / (mask.sum() + 1e-8)

    return loss

LAMBDA_CONTRASTIVE = 0.05
EPOCHS = 7
PATIENCE = 2

optimizer = torch.optim.AdamW(
    model_fusion.parameters(),
    lr=3e-5,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

def run_epoch(loader, training=True):
    if training:
        model_fusion.train()
    else:
        model_fusion.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            ast_features = batch["ast_features"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            with torch.amp.autocast(
                "cuda",
                enabled=torch.cuda.is_available()
            ):
                logits, embeddings = model_fusion(
                    input_ids,
                    attention_mask,
                    ast_features
                )

                cls_loss = F.cross_entropy(logits, labels)
                cont_loss = contrastive_loss(embeddings, labels)

                loss = cls_loss + LAMBDA_CONTRASTIVE * cont_loss

            if training:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item()

            preds = logits.argmax(dim=-1).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.detach().cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="binary",
        zero_division=0
    )

    avg_loss = total_loss / len(loader)

    return avg_loss, accuracy, precision, recall, f1


best_val_f1 = -1
patience_counter = 0
best_state = None

print("\n===== Training CodeT5 + AST Fusion =====")

for epoch in range(EPOCHS):
    train_loss, train_acc, train_prec, train_rec, train_f1 = run_epoch(
        train_loader,
        training=True
    )

    val_loss, val_acc, val_prec, val_rec, val_f1 = run_epoch(
        val_loader,
        training=False
    )

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val Loss  : {val_loss:.4f} | Val F1  : {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {
            k: v.cpu().clone()
            for k, v in model_fusion.state_dict().items()
        }
        patience_counter = 0
        print("New best model saved.")
    else:
        patience_counter += 1

        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

model_fusion.load_state_dict(best_state)

print("\nBest Validation F1:", round(best_val_f1, 4))

model_fusion.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        ast_features = batch["ast_features"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits, embeddings = model_fusion(
            input_ids,
            attention_mask,
            ast_features
        )

        preds = logits.argmax(dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_preds,
    average="binary"
)

cm = confusion_matrix(all_labels, all_preds)

print("\n===== FINAL TEST RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(all_labels, all_preds))

print("\n===== CONFUSION MATRIX =====")
print(cm)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
Device: cuda
Cleaned dataset shape: (488291, 5)
Official train shape: (395477, 5)
Official validation shape: (46355, 5)
Official test shape: (46459, 5)
Reduced train shape: (18000, 5)
label
0    9000
1    9000
Name: count, dtype: int64
language
python    6000
java      6000
cpp       6000
Name: count, dtype: int64


<unknown>:23: SyntaxWarning: invalid escape sequence '\i'
<unknown>:16: SyntaxWarning: invalid decimal literal
<unknown>:24: SyntaxWarning: invalid decimal literal
<unknown>:21: SyntaxWarning: invalid escape sequence '\i'
<unknown>:15: SyntaxWarning: invalid escape sequence '\.'
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:19: SyntaxWarning: invalid escape sequence '\i'
<unknown>:7: SyntaxWarning: invalid decimal literal
<unknown>:9: SyntaxWarning: invalid decimal literal
<unknown>:13: SyntaxWarning: invalid decimal literal
<unknown>:19: SyntaxWarning: invalid escape sequence '\i'
<unknown>:16: SyntaxWarning: invalid escape sequence '\{'
<unknown>:23: SyntaxWarning: invalid escape sequence '\A'
<unknown>:11: SyntaxWarning: invalid decimal literal
<unknown>:11: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:3: SyntaxWarning: invalid escape sequence '\d'
<unknown>:1:

AST features extracted.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
T5EncoderModel LOAD REPORT from: Salesforce/codet5-base
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]


===== Training CodeT5 + AST Fusion =====

Epoch 1/7
Train Loss: 0.3072 | Train F1: 0.9340
Val Loss  : 0.1731 | Val F1  : 0.9577
New best model saved.

Epoch 2/7
Train Loss: 0.1472 | Train F1: 0.9734
Val Loss  : 0.1231 | Val F1  : 0.9659
New best model saved.

Epoch 3/7
Train Loss: 0.0993 | Train F1: 0.9823
Val Loss  : 0.0964 | Val F1  : 0.9750
New best model saved.

Epoch 4/7
Train Loss: 0.0677 | Train F1: 0.9904
Val Loss  : 0.1180 | Val F1  : 0.9694

Epoch 5/7
Train Loss: 0.0539 | Train F1: 0.9925
Val Loss  : 0.0966 | Val F1  : 0.9749
Early stopping triggered.

Best Validation F1: 0.975

===== FINAL TEST RESULTS =====
Accuracy : 0.9768
Precision: 0.9697
Recall   : 0.9827
F1 Score : 0.9762

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.98      0.97      0.98     23952
           1       0.97      0.98      0.98     22507

    accuracy                           0.98     46459
   macro avg       0.98      0.98      0.98    